## STAGE - 1 SCRAPER

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import csv
import os
import re
import time
import random


# ============================================================
# CONFIGURATION
# ============================================================

START_URL = (
    "https://www.sciencedirect.com/"
    "journal/information-and-management/vol/63/issue/8"
)

OUT_FILE = "Information_and_Management_Issues.csv"

MIN_YEAR = 2010


# ============================================================
# DRIVER
# ============================================================

options = webdriver.ChromeOptions()
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)
driver.implicitly_wait(20)


# ============================================================
# HELPERS
# ============================================================

def clean_text(text):
    if not text:
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(text)
    ).strip()


def clean_url(url):
    if not url:
        return ""

    return (
        url
        .split("?")[0]
        .split("#")[0]
        .rstrip("/")
    )


def polite_wait(min_seconds=12, max_seconds=18):
    wait_time = random.uniform(
        min_seconds,
        max_seconds
    )

    print(
        f"Waiting {wait_time:.1f} seconds..."
    )

    time.sleep(wait_time)


def extract_year(text):
    if not text:
        return None

    match = re.search(
        r"\b(19\d{2}|20\d{2})\b",
        text
    )

    if match:
        return int(match.group(1))

    return None


# ============================================================
# BLOCK / ERROR DETECTION
# ============================================================

def blocked_or_not_found():
    """
    Uses visible page text only.
    Avoids false positives from page source / JavaScript.
    """

    try:
        title = clean_text(
            driver.title
        ).lower()
    except Exception:
        title = ""

    try:
        body_text = clean_text(
            driver.find_element(
                By.TAG_NAME,
                "body"
            ).text
        ).lower()
    except Exception:
        body_text = ""

    # --------------------------------------------
    # 404 / missing page
    # --------------------------------------------

    if (
        "page not found" in title
        or "404" in title
        or "page not found" in body_text[:1500]
    ):
        return True

    # --------------------------------------------
    # Visible challenge / block messages
    # --------------------------------------------

    challenge_messages = [
        "verify you are human",
        "are you a human",
        "access denied",
        "unusual traffic",
        "security challenge",
        "please verify",
        "temporarily blocked"
    ]

    for message in challenge_messages:
        if message in body_text[:4000]:
            return True

    return False


def issue_page_loaded():
    """
    Confirms that actual issue content exists.
    """

    # Metadata
    if driver.find_elements(
        By.CLASS_NAME,
        "js-vol-issue"
    ):
        return True

    # Existing ScienceDirect article structure
    if driver.find_elements(
        By.CSS_SELECTOR,
        "li.js-article-list-item dl.js-article"
    ):
        return True

    # Fallback article URLs
    if driver.find_elements(
        By.CSS_SELECTOR,
        "a[href*='/science/article/pii/'], "
        "a[href*='/science/article/abs/pii/']"
    ):
        return True

    return False


# ============================================================
# CSV
# ============================================================

def initialize_csv():

    with open(
        OUT_FILE,
        "w",
        newline="",
        encoding="utf-8"
    ) as file:

        writer = csv.writer(file)

        writer.writerow([
            "Title",
            "URL",
            "Volume Issue",
            "Vol Issue Year"
        ])


def write_rows(rows):

    if not rows:
        return

    with open(
        OUT_FILE,
        "a",
        newline="",
        encoding="utf-8"
    ) as file:

        writer = csv.writer(file)
        writer.writerows(rows)


# ============================================================
# ISSUE METADATA
# ============================================================

def get_issue_metadata():

    vol_issue = "N/A"
    year = None


    # ========================================================
    # VOLUME / ISSUE
    # ========================================================

    try:
        vol_issue = clean_text(
            driver.find_element(
                By.CLASS_NAME,
                "js-vol-issue"
            ).text
        )

    except Exception:

        current_url = driver.current_url

        vol_match = re.search(
            r"/vol/(\d+)",
            current_url,
            re.I
        )

        issue_match = re.search(
            r"/issue/([^/?#]+)",
            current_url,
            re.I
        )

        if vol_match and issue_match:

            vol_issue = (
                f"Volume {vol_match.group(1)}, "
                f"Issue {issue_match.group(1)}"
            )


    # ========================================================
    # YEAR
    # ========================================================

    try:
        issue_status = clean_text(
            driver.find_element(
                By.CLASS_NAME,
                "js-issue-status"
            ).text
        )

        year = extract_year(
            issue_status
        )

    except Exception:
        pass


    # --------------------------------------------------------
    # FALLBACK YEAR
    # --------------------------------------------------------

    if year is None:

        try:
            body_text = clean_text(
                driver.find_element(
                    By.TAG_NAME,
                    "body"
                ).text[:4000]
            )

            year = extract_year(
                body_text
            )

        except Exception:
            pass


    return vol_issue, year


# ============================================================
# ARTICLE EXTRACTION
# ============================================================

def extract_articles():

    results = []

    seen_urls = set()


    # ========================================================
    # METHOD 1
    # Existing ScienceDirect structure
    # ========================================================

    articles = driver.find_elements(
        By.CSS_SELECTOR,
        "li.js-article-list-item dl.js-article"
    )


    for article in articles:

        try:

            link = article.find_element(
                By.CSS_SELECTOR,
                "a.article-content-title"
            )

            article_url = clean_url(
                link.get_attribute(
                    "href"
                )
            )

            article_title = clean_text(
                article.find_element(
                    By.CLASS_NAME,
                    "js-article-title"
                ).text
            )


            if not article_url:
                continue

            if not article_title:
                continue

            if article_url in seen_urls:
                continue


            seen_urls.add(
                article_url
            )


            results.append([
                article_title,
                article_url
            ])


        except Exception:
            continue


    # ========================================================
    # METHOD 2
    # Fallback for newer ScienceDirect markup
    # ========================================================

    if not results:

        links = driver.find_elements(
            By.CSS_SELECTOR,
            "a[href*='/science/article/pii/'], "
            "a[href*='/science/article/abs/pii/']"
        )


        for link in links:

            try:

                article_url = clean_url(
                    link.get_attribute(
                        "href"
                    )
                )

                article_title = clean_text(
                    link.text
                )


                if not article_url:
                    continue

                if not article_title:
                    continue

                if len(article_title) < 8:
                    continue


                bad_titles = {
                    "pdf",
                    "view pdf",
                    "download pdf",
                    "abstract",
                    "show abstract",
                    "view abstract",
                    "full text",
                    "view full text",
                    "article"
                }


                if (
                    article_title.lower()
                    in bad_titles
                ):
                    continue


                if article_url in seen_urls:
                    continue


                seen_urls.add(
                    article_url
                )


                results.append([
                    article_title,
                    article_url
                ])


            except Exception:
                continue


    return results


# ============================================================
# PREVIOUS VOLUME / ISSUE
# ============================================================

def get_previous_issue_url():

    # ========================================================
    # PRIMARY SELECTOR
    # ========================================================

    try:

        prev_element = driver.find_element(
            By.CSS_SELECTOR,
            "nav.issue-navigation "
            "div.navigation-pre "
            "a.text-m"
        )

        href = clean_url(
            prev_element.get_attribute(
                "href"
            )
        )

        if href:
            return href

    except Exception:
        pass


    # ========================================================
    # FALLBACK SELECTORS
    # ========================================================

    selectors = [
        "a[aria-label*='Previous']",
        "a[title*='Previous']",
        "nav.issue-navigation a"
    ]


    for selector in selectors:

        try:

            elements = driver.find_elements(
                By.CSS_SELECTOR,
                selector
            )


            for element in elements:

                text = clean_text(
                    element.text
                ).lower()

                aria = clean_text(
                    element.get_attribute(
                        "aria-label"
                    )
                ).lower()

                title = clean_text(
                    element.get_attribute(
                        "title"
                    )
                ).lower()


                combined = (
                    text
                    + " "
                    + aria
                    + " "
                    + title
                )


                if (
                    "previous" not in combined
                    and "prev" not in combined
                ):
                    continue


                href = clean_url(
                    element.get_attribute(
                        "href"
                    )
                )


                if href:
                    return href


        except Exception:
            continue


    return None


# ============================================================
# MAIN STAGE 1
# ============================================================

def scrape_stage_1():

    initialize_csv()

    current_url = START_URL

    seen_issue_urls = set()
    seen_article_urls = set()

    total_issues = 0
    total_articles = 0


    print("=" * 70)
    print("INFORMATION & MANAGEMENT")
    print("STAGE 1")
    print("Starting from Volume 63 Issue 8")
    print("=" * 70)


    # ========================================================
    # LOOP THROUGH PREVIOUS ISSUES
    # ========================================================

    while current_url:


        # ----------------------------------------------------
        # LOOP PROTECTION
        # ----------------------------------------------------

        if current_url in seen_issue_urls:

            print(
                "\nIssue already visited."
            )

            print(
                "Stopping to prevent infinite loop."
            )

            break


        seen_issue_urls.add(
            current_url
        )


        print(
            "\n" + "=" * 70
        )

        print(
            "Opening:"
        )

        print(
            current_url
        )

        print(
            "=" * 70
        )


        try:

            # =================================================
            # OPEN PAGE
            # =================================================

            driver.get(
                current_url
            )


            WebDriverWait(
                driver,
                30
            ).until(
                EC.presence_of_element_located(
                    (By.TAG_NAME, "body")
                )
            )


            # One deliberate wait per page
            polite_wait(
                12,
                18
            )


            # =================================================
            # BLOCK / ERROR CHECK
            # =================================================

            if blocked_or_not_found():

                # Sometimes challenge-related words may appear
                # even though issue content loaded.
                if issue_page_loaded():

                    print(
                        "Challenge-related text detected, "
                        "but issue content loaded."
                    )

                    print(
                        "Continuing..."
                    )

                else:

                    print(
                        "ScienceDirect returned an actual "
                        "blocked/error page."
                    )

                    print(
                        "Stopping safely."
                    )

                    break


            # =================================================
            # GET ISSUE METADATA
            # =================================================

            vol_issue, year = (
                get_issue_metadata()
            )


            print(
                "Volume Issue:",
                vol_issue
            )

            print(
                "Year:",
                year
            )


            # =================================================
            # STOP BEFORE 2010
            # =================================================

            if (
                year is not None
                and year < MIN_YEAR
            ):

                print(
                    f"\nReached {year}."
                )

                print(
                    f"Stopping because "
                    f"MIN_YEAR = {MIN_YEAR}."
                )

                break


            # =================================================
            # EXTRACT ARTICLES
            # =================================================

            articles = (
                extract_articles()
            )


            print(
                "Articles detected:",
                len(articles)
            )


            rows = []


            for (
                article_title,
                article_url
            ) in articles:


                # ---------------------------------------------
                # GLOBAL DUPLICATE CHECK
                # ---------------------------------------------

                if (
                    article_url
                    in seen_article_urls
                ):
                    continue


                seen_article_urls.add(
                    article_url
                )


                rows.append([
                    article_title,
                    article_url,
                    vol_issue,
                    year if year else "N/A"
                ])


            # =================================================
            # WRITE IMMEDIATELY
            # =================================================

            write_rows(
                rows
            )


            total_issues += 1

            total_articles += len(
                rows
            )


            print(
                "Articles saved:",
                len(rows)
            )

            print(
                "Running article total:",
                total_articles
            )


            # =================================================
            # FIND PREVIOUS ISSUE
            # =================================================

            previous_url = (
                get_previous_issue_url()
            )


            if not previous_url:

                print(
                    "\nNo previous volume/issue "
                    "link found."
                )

                break


            if (
                previous_url
                in seen_issue_urls
            ):

                print(
                    "\nPrevious issue has "
                    "already been visited."
                )

                break


            print(
                "Next previous issue:"
            )

            print(
                previous_url
            )


            # The next loop performs the wait
            # after loading the next page.

            current_url = (
                previous_url
            )


        except Exception as e:

            print(
                "\nERROR:"
            )

            print(
                e
            )

            break


    # ========================================================
    # COMPLETE
    # ========================================================

    print(
        "\n" + "=" * 70
    )

    print(
        "STAGE 1 COMPLETE"
    )

    print(
        "=" * 70
    )

    print(
        "Issues processed:",
        total_issues
    )

    print(
        "Unique articles saved:",
        total_articles
    )

    print(
        "Output file:",
        os.path.abspath(
            OUT_FILE
        )
    )


# ============================================================
# RUN
# ============================================================

try:

    scrape_stage_1()

finally:

    driver.quit()